# PC Members Extraction

- Download and parse PC pages. 
- Each page is saved locally and then converted into a structured table.

## 1. Setup

In [38]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import date
from pathlib import Path

In [56]:
ROOT = Path.cwd().parent

## 2. Scraping and Parsing Utilities

In [ ]:
def parse_committee_html(html, url, year):
    """
    Parse a PC page (POPL-style, conf.researchr layout)
    and extract structured information about PC members.
    
    The function assumes the typical conf.researchr structure where each
    committee member appears as a profile link containing:
    
    - an <h3> element with the person's name (and possibly role in <small>)
    - an <h4><span> with the affiliation
    - an <h4><small> with the country
    
    Parameters
    ----------
    html : str
        The raw HTML content of the committee page (loaded from a
        saved snapshot).
    url : str
        The original page URL, used to build profile links.
    year : int
        The conference year.
    
    Returns
    -------
    pd.DataFrame
        One row per committee member with the columns:
        year, conference, name, role, affiliation, country, person_url.
    
    -----
    This parser is based on conf.researchr-based conference pages.
    If the website layout changes, the selectors may need to be updated.
    Missing affiliation or country fields are returned as empty strings.
    
    """
    
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    for a in soup.select('a[href*="/profile/"]'):
        h3 = a.select_one("h3.media-heading")
        if not h3:
            continue
        
        name = h3.contents[0].strip()

        role_tag = h3.select_one("small")
        if role_tag:
            role_text = role_tag.get_text(strip=True).lower()
            if "associate" in role_text and "chair" in role_text:
                role = "Associate Chair"
            elif "chair" in role_text:
                role = "PC Chair"
            else:
                role = "Committee Member"
        else:
            role = "Committee Member"

        aff_tag = a.select_one("h4 span.text-black")
        affiliation = aff_tag.get_text(strip=True) if aff_tag else ""

        country_tag = a.select("h4 small")
        country = country_tag[0].get_text(strip=True) if country_tag else ""

        rows.append({
            "year": year,
            "conference": "POPL",
            "name": name,
            "role": role,
            "affiliation": affiliation,
            "country": country,
            "person_url": urljoin(url, a["href"]),
        })

    return pd.DataFrame(rows)

In [ ]:
def scrape_conference(conference_name, url_dict, repo_root=ROOT):
    """
    Download and parse PC pages for a conference.
    
    For each year, this function fetches the committee page (aka PC), saves a
    dated HTML snapshot under data/raw/ (so we have a frozen copy),
    and then parses it into a structured dataframe.
    
    This keeps the raw source material reproducible while giving us a
    clean table of committee members for analysis.
    
    Parameters
    ----------
    conference_name : str
        Conference short name (e.g., "POPL", "ICFP").
    url_dict : dict
        Mapping from year to committee page URL.
    repo_root : Path
        Project root directory.
    
    Returns
    -------
    pd.DataFrame
        A combined dataframe with one row per committee member
        across the specified years.
    """

    raw_dir = ROOT / "data" / "raw" / f"{conference_name.lower()}_pc_htmls"
    raw_dir.mkdir(parents=True, exist_ok=True)

    today = date.today().isoformat()
    all_dfs = []

    for year, url in url_dict.items():
        print(f"{conference_name} {year}...")

        #download
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        #save snapshot
        file_path = raw_dir / f"{conference_name.lower()}{year}_program_committee_{today}.html"
        file_path.write_text(response.text, encoding="utf-8")

        #parse from saved file
        html = file_path.read_text(encoding="utf-8")
        df_year = parse_committee_html(html, url, year)

        df_year["conference"] = conference_name
        all_dfs.append(df_year)

    df_all = pd.concat(all_dfs, ignore_index=True)

    return df_all

In [49]:
def save_parquet(df, relative_path):
    path = ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)

## 3. POPL PC Dataset

In [ ]:
urls_popl = {
 2017: 'https://popl17.sigplan.org/committee/popl-2017-papers-program-committee',
 2018: 'https://popl18.sigplan.org/committee/popl-2018-papers-program-committee',
 2019: 'https://popl19.sigplan.org/committee/popl-2019-research-papers-program-committee',
 2020: 'https://popl20.sigplan.org/committee/popl-2020-papers-program-committee',
 2021: 'https://popl21.sigplan.org/committee/POPL-2021-research-papers-program-committee',
 2022: 'https://popl22.sigplan.org/committee/POPL-2022-popl-research-papers-program-committee',
 2023: 'https://popl23.sigplan.org/committee/POPL-2023-popl-research-papers-program-committee',
 2024: 'https://popl24.sigplan.org/committee/POPL-2024-popl-research-papers-program-committee',
 2025: 'https://popl25.sigplan.org/committee/POPL-2025-popl-research-papers-program-committee'
}

In [40]:
df_popl = scrape_conference("POPL", urls_popl)
df_popl.head()

POPL 2017...
POPL 2018...
POPL 2019...
POPL 2020...
POPL 2021...
POPL 2022...
POPL 2023...
POPL 2024...
POPL 2025...


,year,conference,name,role,affiliation,country,person_url
0,2017,POPL,Andrew D. Gordon,PC Chair,Microsoft Research and University of Edinburgh,United Kingdom,https://popl17.sigplan.org/profile/andrewdgordon
1,2017,POPL,Martin Abadi,Committee Member,Google,,https://popl17.sigplan.org/profile/martinabadi
2,2017,POPL,Josh Berdine,Committee Member,Facebook,,https://popl17.sigplan.org/profile/joshberdine
3,2017,POPL,Johannes Borgström,Committee Member,Uppsala University,,https://popl17.sigplan.org/profile/johannesbor...
4,2017,POPL,Avik Chaudhuri,Committee Member,Facebook,United States,https://popl17.sigplan.org/profile/avikchaudhuri


In [43]:
print(df_popl.groupby("year").size())
print(df_popl.isna().sum())
df_popl.sample(10)

year
2017    29
2018    52
2019    52
2020    54
2021    52
2022    56
2023    84
2024    91
2025    82
dtype: int64
year           0
conference     0
name           0
role           0
affiliation    0
country        0
person_url     0
dtype: int64


,year,conference,name,role,affiliation,country,person_url
279,2022,POPL,François Pottier,Committee Member,Inria,France,https://popl22.sigplan.org/profile/francoispot...
271,2022,POPL,Fan Long,Committee Member,"University of Toronto, Canada",,https://popl22.sigplan.org/profile/fanlong
76,2018,POPL,Philip Wadler,Committee Member,"University of Edinburgh, UK",United Kingdom,https://popl18.sigplan.org/profile/philipwadler
381,2024,POPL,Alexandra Silva,Associate Chair,Cornell University,United States,https://popl24.sigplan.org/profile/alexandrasilva
140,2020,POPL,Arjun Guha,Committee Member,University of Massachusetts Amherst,,https://popl20.sigplan.org/profile/arjunguha
265,2022,POPL,Neel Krishnaswami,Committee Member,University of Cambridge,United Kingdom,https://popl22.sigplan.org/profile/neelakantan...
517,2025,POPL,J. Garrett Morris,Committee Member,University of Iowa,United States,https://popl25.sigplan.org/profile/jgarrettmorris
258,2022,POPL,Justin Hsu,Committee Member,Cornell University,United States,https://popl22.sigplan.org/profile/justinhsu
123,2019,POPL,Robbert Krebbers,Committee Member,Delft University of Technology,Netherlands,https://popl19.sigplan.org/profile/robbertkreb...
158,2020,POPL,Jeffrey S. Foster,Committee Member,Tufts University,United States,https://popl20.sigplan.org/profile/jeffreysfoster


In [55]:
save_parquet(df_popl, ROOT / "data" / "intermediate" / "popl_program_committee.parquet")

## 4. ICFP PC Dataset

In [57]:
urls_icfp = {
 2017: 'https://icfp17.sigplan.org/committee/icfp-2017-papers-program-committee',
 2018: 'https://icfp18.sigplan.org/committee/icfp-2018-papers-program-committee',
 2019: 'https://icfp19.sigplan.org/committee/icfp-2019-papers-program-committee',
 2020: 'https://icfp20.sigplan.org/committee/icfp-2020-papers-program-committee',
 2021: 'https://icfp21.sigplan.org/committee/icfp-2021-papers-program-committee',
 2022: 'https://icfp22.sigplan.org/committee/icfp-2022-papers-program-committee',
 2023: 'https://icfp23.sigplan.org/committee/icfp-2023-papers-program-committee',
 2024: 'https://icfp24.sigplan.org/committee/icfp-2024-papers-icfp-papers-and-events',
 2025: 'https://icfp25.sigplan.org/committee/icfp-2025-papers-icfp-papers-and-events'
}

In [58]:
df_icfp = scrape_conference("ICFP", urls_icfp)
df_icfp.head()

ICFP 2017...
ICFP 2018...
ICFP 2019...
ICFP 2020...
ICFP 2021...
ICFP 2022...
ICFP 2023...
ICFP 2024...
ICFP 2025...


,year,conference,name,role,affiliation,country,person_url
0,2017,ICFP,Mark Jones,PC Chair,Portland State University,,https://icfp17.sigplan.org/profile/markjones
1,2017,ICFP,Robert Atkey,Committee Member,University of Strathclyde,,https://icfp17.sigplan.org/profile/robertatkey
2,2017,ICFP,Adam Chlipala,Committee Member,"Massachusetts Institute of Technology, USA",United States,https://icfp17.sigplan.org/profile/adamchlipala
3,2017,ICFP,Dominique Devriese,Committee Member,"KU Leuven, Belgium",Belgium,https://icfp17.sigplan.org/profile/dominiquede...
4,2017,ICFP,Martin Erwig,Committee Member,Oregon State University,United States,https://icfp17.sigplan.org/profile/martinerwig


In [63]:
print(df_icfp.groupby("year").size())
print(df_icfp.isna().sum())

year
2017    23
2018    18
2019    20
2020    17
2021    30
2022    42
2023    54
2024    51
2025    63
dtype: int64
year           0
conference     0
name           0
role           0
affiliation    0
country        0
person_url     0
dtype: int64


In [64]:
df_icfp.head()

,year,conference,name,role,affiliation,country,person_url
0,2017,ICFP,Mark Jones,PC Chair,Portland State University,,https://icfp17.sigplan.org/profile/markjones
1,2017,ICFP,Robert Atkey,Committee Member,University of Strathclyde,,https://icfp17.sigplan.org/profile/robertatkey
2,2017,ICFP,Adam Chlipala,Committee Member,"Massachusetts Institute of Technology, USA",United States,https://icfp17.sigplan.org/profile/adamchlipala
3,2017,ICFP,Dominique Devriese,Committee Member,"KU Leuven, Belgium",Belgium,https://icfp17.sigplan.org/profile/dominiquede...
4,2017,ICFP,Martin Erwig,Committee Member,Oregon State University,United States,https://icfp17.sigplan.org/profile/martinerwig


In [78]:
save_parquet(df_icfp, ROOT / "data" / "intermediate" / "ifcp_program_committee.parquet")

## 5. OOPSLA PC Dataset

In [65]:
urls_oopsla= {
 2017: 'https://2017.splashcon.org/committee/splash-2017-oopsla-program-committee',
 2018: 'https://2018.splashcon.org/committee/splash-2018-oopsla-committee',
 2019: 'https://2019.splashcon.org/committee/splash-2019-oopsla-review-committee',
 2020: 'https://2020.splashcon.org/committee/splash-2020-oopsla-review-committee',
 2021: 'https://2021.splashcon.org/committee/splash-2021-oopsla-review-committee',
 2022: 'https://2022.splashcon.org/committee/splash-2022-psla-review-committee',
 2023: 'https://2023.splashcon.org/committee/splash-2023-oopsla-review-committee',
 2024: 'https://2024.splashcon.org/committee/splash-2024-papers-review-committee',
 2025: 'https://2025.splashcon.org/committee/splash-2025-OOPSLA-oopsla-review-committee'
}

In [66]:
df_oopsala = scrape_conference("OOPSLA", urls_oopsla)
df_oopsala.head()

OOPSLA 2017...
OOPSLA 2018...
OOPSLA 2019...
OOPSLA 2020...
OOPSLA 2021...
OOPSLA 2022...
OOPSLA 2023...
OOPSLA 2024...
OOPSLA 2025...


,year,conference,name,role,affiliation,country,person_url
0,2017,OOPSLA,Jonathan Aldrich,PC Chair,Carnegie Mellon University,United States,https://2017.splashcon.org/profile/jonathanald...
1,2017,OOPSLA,Suparna Bhattacharya,Committee Member,Hewlett-Packard Enterprise,India,https://2017.splashcon.org/profile/suparnabhat...
2,2017,OOPSLA,Hans-J. Boehm,Committee Member,Google,United States,https://2017.splashcon.org/profile/hansjboehm
3,2017,OOPSLA,Viviana Bono,Committee Member,University of Torino,Italy,https://2017.splashcon.org/profile/vivianabono
4,2017,OOPSLA,Kim Bruce,Committee Member,Pomona College,United States,https://2017.splashcon.org/profile/kimbruce


In [90]:
df_oopsala.query('name.str.contains("Eelco")')

,year,conference,name,role,affiliation,country,person_url
59,2018,OOPSLA,Eelco Visser,Committee Member,Delft University of Technology,Netherlands,https://2018.splashcon.org/profile/eelcovisser
61,2019,OOPSLA,Eelco Visser,Committee Member,Delft University of Technology,Netherlands,https://2019.splashcon.org/profile/eelcovisser
120,2020,OOPSLA,Eelco Visser,Committee Member,Delft University of Technology,,https://2020.splashcon.org/profile/eelcovisser


In [ ]:
df_oopsala.loc[61,'role'] = "PC Chair" #manual fix.

In [98]:
save_parquet(df_oopsala, ROOT / "data" / "intermediate" / "oopsala_program_committee.parquet")

## Checking PC lists

- Manually checked and updated for Eelco Visser role in 2019.
- Want to understand whether **name** is a unique identifier.

In [391]:
df = pd.concat([df_popl,df_icfp,df_oopsala]).reset_index(drop=True)

In [394]:
save_parquet(df, ROOT / "data" / "intermediate" / "program_committee_members.parquet")

In [395]:
df.head(2)

,year,conference,name,role,affiliation,country,person_url
0,2017,POPL,Andrew D. Gordon,PC Chair,Microsoft Research and University of Edinburgh,United Kingdom,https://popl17.sigplan.org/profile/andrewdgordon
1,2017,POPL,Martin Abadi,Committee Member,Google,,https://popl17.sigplan.org/profile/martinabadi


In [396]:
len(set(df.name))

739

In [397]:
split_data = df['person_url'].str.split('/profile/', expand=True)
len(set(split_data[1]))

743

In [398]:
df['id'] = split_data[1]

In [399]:
df.drop_duplicates(subset='id',keep='first')['name'].value_counts()

name
Nobuko Yoshida       2
Constantin Enea      2
Filip Sieczkowski    2
Shaz Qadeer          2
Andrew D. Gordon     1
                    ..
Yakir Vizel          1
Wenxi Wang           1
Christian Wimmer     1
Jialu Zhang          1
Xiangyu Zhang        1
Name: count, Length: 739, dtype: int64

In [400]:
dif = ["Nobuko Yoshida","Constantin Enea","Filip Sieczkowski","Shaz Qadeer"]

In [401]:
df.head(2)

,year,conference,name,role,affiliation,country,person_url,id
0,2017,POPL,Andrew D. Gordon,PC Chair,Microsoft Research and University of Edinburgh,United Kingdom,https://popl17.sigplan.org/profile/andrewdgordon,andrewdgordon
1,2017,POPL,Martin Abadi,Committee Member,Google,,https://popl17.sigplan.org/profile/martinabadi,martinabadi


In [402]:
df.drop_duplicates('id').query("name in @dif").sort_values(by='id')

,year,conference,name,role,affiliation,country,person_url,id
44,2018,POPL,Constantin Enea,Committee Member,Université Paris Diderot,France,https://popl18.sigplan.org/profile/constantinenea,constantinenea
1313,2025,OOPSLA,Constantin Enea,Committee Member,"LIX, CNRS, Ecole Polytechnique",,https://2025.splashcon.org/profile/constantine...,constantinenea1
282,2022,POPL,Filip Sieczkowski,Committee Member,University of Wrocław,Poland,https://popl22.sigplan.org/profile/filipsieczk...,filipsieczkowski
796,2024,ICFP,Filip Sieczkowski,Committee Member,,,https://icfp24.sigplan.org/profile/filipsieczk...,filipsieczkowski1
27,2017,POPL,Nobuko Yoshida,Committee Member,"Imperial College London, UK",,https://popl17.sigplan.org/profile/nobukoyoshida,nobukoyoshida
806,2024,ICFP,Nobuko Yoshida,Committee Member,University of Oxford,United Kingdom,https://icfp24.sigplan.org/profile/nobukoyoshida1,nobukoyoshida1
1248,2024,OOPSLA,Shaz Qadeer,Committee Member,Facebook,United States,https://2024.splashcon.org/profile/shazqadeer,shazqadeer
527,2025,POPL,Shaz Qadeer,Committee Member,"Meta, Inc.",United States,https://popl25.sigplan.org/profile/shazqadeer1,shazqadeer1


In [403]:
df[['name','person_url','id']].head(3)

,name,person_url,id
0,Andrew D. Gordon,https://popl17.sigplan.org/profile/andrewdgordon,andrewdgordon
1,Martin Abadi,https://popl17.sigplan.org/profile/martinabadi,martinabadi
2,Josh Berdine,https://popl17.sigplan.org/profile/joshberdine,joshberdine


In [255]:
df_names = df[['name','id']].drop_duplicates('id').reset_index(drop=True)
df_names.head()

,name,id
0,Andrew D. Gordon,andrewdgordon
1,Martin Abadi,martinabadi
2,Josh Berdine,joshberdine
3,Johannes Borgström,johannesborgstrom
4,Avik Chaudhuri,avikchaudhuri


In [256]:
df_names.shape

(743, 2)